In [26]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torch

In [27]:
class VggFeature(nn.Module):
    def __init__(self, drop = 0.2):
        super().__init__()

        # Convolution
        self.conv1a = nn.Conv2d(in_channels=1, out_channels = 32, kernel_size=3, padding=1)
        self.conv1b = nn.Conv2d(in_channels=32, out_channels = 32, kernel_size=3, padding=1)

        self.conv2a = nn.Conv2d(in_channels=32, out_channels = 64, kernel_size=3, padding=1)
        self.conv2b = nn.Conv2d(in_channels=64, out_channels = 64, kernel_size=3, padding=1)

        self.conv3a = nn.Conv2d(in_channels=64, out_channels = 128, kernel_size=3, padding=1)
        self.conv3b = nn.Conv2d(in_channels=128, out_channels = 128, kernel_size=3, padding=1)

        self.conv4a = nn.Conv2d(in_channels=128, out_channels = 256, kernel_size=3, padding=1)
        self.conv4b = nn.Conv2d(in_channels=256, out_channels = 256, kernel_size=3, padding=1)

        # Pool
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Batch Normalization
        self.bn1a = nn.BatchNorm2d(32)
        self.bn1b = nn.BatchNorm2d(32)

        self.bn2a = nn.BatchNorm2d(64)
        self.bn2b = nn.BatchNorm2d(64)

        self.bn3a = nn.BatchNorm2d(128)
        self.bn3b = nn.BatchNorm2d(128)

        self.bn4a = nn.BatchNorm2d(256)
        self.bn4b = nn.BatchNorm2d(256)

        # Flatten
        self.lin1 = nn.Linear(256 * 2 * 2, 2048)
        
        # Dropout
        self.drop = nn.Dropout(p=drop)

    def forward(self, x):
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = F.relu(self.bn1b(self.conv1b(x)))
        x = self.pool(x)

        x = F.relu(self.bn2a(self.conv2a(x)))
        x = F.relu(self.bn2b(self.conv2b(x)))
        x = self.pool(x)

        x = F.relu(self.bn3a(self.conv3a(x)))
        x = F.relu(self.bn3b(self.conv3b(x)))
        x = self.pool(x)

        x = F.relu(self.bn4a(self.conv4a(x)))
        x = F.relu(self.bn4b(self.conv4b(x)))
        x = self.pool(x)

        x = x.view(-1, 256 * 2 * 2)
        x = F.relu(self.drop(self.lin1(x)))

        return x

In [28]:
class Vgg(VggFeature):
    def __init__(self, drop=0.2):
        super().__init__(drop)
        self.lin2 = nn.Linear(2048, 7)

    def forward(self, x):
        x = super().forward(x)
        x = self.lin2(x)
        return x

In [50]:
x = torch.rand((32,1,48,48))

In [51]:
vggFeature = VggFeature(drop = 0.2)

In [52]:
vggModel = Vgg(drop = 0.2)

In [53]:
a = vggModel.forward(x)

In [54]:
a.shape

torch.Size([72, 7])

In [55]:
a

tensor([[-0.5461,  0.2293,  0.3253, -0.4943,  0.2345,  0.2574, -0.3553],
        [-0.3646,  0.1939,  0.1039,  0.2804,  0.0685,  0.3410,  0.0681],
        [-0.3409,  0.1155,  0.1262, -0.0428,  0.0460,  0.1619, -0.1886],
        [-0.2410,  0.1564,  0.4858,  0.0498,  0.2418,  0.4764, -0.2706],
        [-0.3744, -0.4073,  0.4495,  0.2878,  0.0050, -0.0386, -0.3006],
        [-0.1020, -0.0519,  0.3270, -0.0188, -0.0865,  0.4976,  0.0174],
        [-0.4984,  0.1941,  0.5825,  0.1840,  0.1017,  0.3340,  0.0770],
        [-0.5134,  0.5174,  0.1490,  0.0151,  0.1141,  0.2585, -0.1002],
        [-0.5623,  0.3385,  0.7206, -0.0756,  0.1498,  0.2325, -0.0217],
        [-0.2965,  0.1004,  0.0620,  0.0686,  0.0482,  0.0998, -0.1367],
        [-0.0295,  0.6446,  0.2910,  0.5834, -0.2265,  0.2510,  0.1823],
        [-0.6268,  0.2449,  0.2173,  0.4555, -0.2331,  0.5945, -0.1372],
        [ 0.0216, -0.0785,  0.1299,  0.1043,  0.0060,  0.2648, -0.0735],
        [-0.3235,  0.0343, -0.3443,  0.0919,  0.157